# Transfer Learning with DenseNet121

This is the main model. The choice of DenseNet121 is grounded in CheXNet (Rajpurkar et al., Stanford 2017), which used the same backbone for pneumonia detection on chest radiographs and showed it outperforms a from-scratch CNN on this kind of imagery.

Training proceeds in two stages, which is the standard recipe:

**Stage 1 (feature extraction).** Freeze the entire DenseNet backbone and train only the new classifier head with a normal learning rate (1e-3). This lets the head adapt to the pneumonia task without disturbing the pretrained convolutional features.

**Stage 2 (fine-tuning).** Unfreeze the top 30 layers of the backbone and continue training with a much smaller learning rate (1e-5). The small LR is critical: the pretrained weights already encode useful features, and a large LR would overwrite them.

The best checkpoint becomes `models/best_model.keras`, which the Streamlit app loads.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from src.data_loader import build_datasets
from src.model_builder import build_densenet121, unfreeze_top_layers, standard_callbacks

tf.keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__, " GPU:", bool(tf.config.list_physical_devices("GPU")))

## 1. Build pipeline and model

In [ ]:
DATA_DIR = PROJECT_ROOT / "chest_xray"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIG_DIR = REPORTS_DIR / "figures"
MODELS_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

train_ds, val_ds, test_ds, class_weights, manifest = build_datasets(DATA_DIR)
print("Class weights:", class_weights)

In [ ]:
model = build_densenet121()
trainable_params = sum(np.prod(v.shape) for v in model.trainable_weights)
total_params = model.count_params()
print(f"Total params:     {total_params:,}")
print(f"Trainable (head): {trainable_params:,}")

## 2. Stage 1: feature extraction

Backbone is frozen. Only the head trains. Roughly 10 epochs is enough; early stopping handles the rest.

In [ ]:
stage1_ckpt = str(MODELS_DIR / "densenet_stage1.keras")
stage1_callbacks = standard_callbacks(stage1_ckpt, patience=4)

history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weights,
    callbacks=stage1_callbacks,
    verbose=1,
)

## 3. Stage 2: fine-tuning

Unfreeze the top 30 layers, drop the learning rate to 1e-5, continue training. The final best checkpoint is saved as `best_model.keras`.

In [ ]:
model = unfreeze_top_layers(model, num_layers=30, learning_rate=1e-5)
trainable_params = sum(np.prod(v.shape) for v in model.trainable_weights)
print(f"Trainable params after unfreezing: {trainable_params:,}")

In [ ]:
best_ckpt = str(MODELS_DIR / "best_model.keras")
stage2_callbacks = standard_callbacks(best_ckpt, patience=5)

history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=stage2_callbacks,
    verbose=1,
)

## 4. Combined training curves

In [ ]:
h1 = pd.DataFrame(history_stage1.history).assign(stage="stage1_features")
h2 = pd.DataFrame(history_stage2.history).assign(stage="stage2_finetune")
h2.index = h2.index + len(h1)
hist = pd.concat([h1, h2])
hist.to_csv(REPORTS_DIR / "densenet_history.csv", index=False)

boundary = len(h1) - 0.5
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(hist.index, hist["loss"], label="train")
axes[0].plot(hist.index, hist["val_loss"], label="val")
axes[0].axvline(boundary, color="gray", linestyle=":", label="unfreeze top 30")
axes[0].set_title("DenseNet121: loss")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend()

axes[1].plot(hist.index, hist["auc"], label="train AUC")
axes[1].plot(hist.index, hist["val_auc"], label="val AUC")
axes[1].plot(hist.index, hist["recall"], label="train recall", linestyle="--")
axes[1].plot(hist.index, hist["val_recall"], label="val recall", linestyle="--")
axes[1].axvline(boundary, color="gray", linestyle=":")
axes[1].set_title("DenseNet121: AUC and recall")
axes[1].set_xlabel("epoch"); axes[1].legend()

fig.tight_layout()
fig.savefig(FIG_DIR / "densenet_training_curves.png", bbox_inches="tight")
plt.show()

## 5. Quick test set check

Headline metrics now; full evaluation lives in the next notebook.

In [ ]:
best = tf.keras.models.load_model(best_ckpt)
test_metrics = best.evaluate(test_ds, return_dict=True, verbose=1)
test_metrics = {k: float(v) for k, v in test_metrics.items()}
(REPORTS_DIR / "densenet_test_metrics.json").write_text(json.dumps(test_metrics, indent=2))
print(test_metrics)
print("Best model saved at:", best_ckpt)